# OpenTracy - Complete Test Notebook

This notebook tests all functionalities of the `opentracy` package:

1. **Hub**: Download manager (like NLTK, spaCy, HuggingFace)
2. **Providers**: OpenAI, Anthropic, Google, Groq, Mistral, vLLM
3. **Semantic Routing**: Load router and route prompts
4. **Training**: Train custom clusters and profile models

## 1. Installation & Setup

In [ ]:
import subprocess
import sys

# Install opentracy from parent directory
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".."])

In [ ]:
import os
from pathlib import Path

# Set your API keys here or use environment variables
os.environ["OPENAI_API_KEY"] = ""
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["GOOGLE_API_KEY"] = "..."
# os.environ["GROQ_API_KEY"] = "gsk_..."
# os.environ["MISTRAL_API_KEY"] = "..."

In [ ]:
# Test imports
from opentracy import (
    # Version
    __version__,
    # Clients
    create_client,
    OpenAIClient,
    AnthropicClient,
    GoogleClient,
    GroqClient,
    MistralClient,
    VLLMClient,
    MockLLMClient,
    # Router
    load_router,
    UniRouteRouter,
    # Hub (like NLTK/spaCy/HuggingFace)
    download,
    list_packages,
    package_info,
    remove,
    path,
    Hub,
    OPENTRACY_DATA_HOME,
)

print(f"OpenTracy v{__version__} imported successfully!")
print(f"Data directory: {OPENTRACY_DATA_HOME}")

## 2. Hub - Package Manager

OpenTracy uses a download manager similar to NLTK, spaCy, and HuggingFace.

**CLI Commands:**
```bash
opentracy list                      # List all packages
opentracy download weights-mmlu-v1  # Download a package
opentracy info weights-mmlu-v1      # Show package info
opentracy path weights-mmlu-v1      # Show installation path
opentracy remove weights-mmlu-v1    # Remove a package
opentracy verify weights-mmlu-v1    # Verify package integrity
```

**Python API:**
```python
import opentracy

opentracy.download("weights-mmlu-v1")  # Download
opentracy.list_packages()               # List
opentracy.package_info("weights-mmlu-v1")  # Info
opentracy.path("weights-mmlu-v1")       # Path
opentracy.remove("weights-mmlu-v1")     # Remove
```

In [ ]:
# List available packages
hub = Hub()
packages = hub.list_packages()

print("Available Packages")
print("=" * 90)
print(f"{'ID':<25} {'Version':<10} {'Source':<12} {'Status':<15} Description")
print("-" * 90)

for pkg in packages:
    status = "installed" if hub.is_installed(pkg.id) else "not installed"
    source = getattr(pkg, 'source_type', 'huggingface')
    desc = pkg.description[:30] + "..." if len(pkg.description) > 30 else pkg.description
    print(f"{pkg.id:<25} {pkg.version:<10} {source:<12} {status:<15} {desc}")

print(f"\nTotal: {len(packages)} packages")
print(f"Data directory: {hub.data_home}")

In [ ]:
# Get info about a specific package
info = package_info("weights-mmlu-v1")

if info:
    print(f"Package: {info['name']} ({info['id']})")
    print("=" * 50)
    print(f"Version:     {info['version']}")
    print(f"Category:    {info['category']}")
    print(f"Description: {info['description']}")
    print(f"Author:      {info['author']}")
    print(f"License:     {info['license']}")
    print(f"Source:      {info.get('source_type', 'huggingface')}")
    
    if info.get('hf_repo_id'):
        print(f"HF Repo:     {info['hf_repo_id']}")
    
    if info.get('size_bytes'):
        size_mb = info['size_bytes'] / (1024 * 1024)
        print(f"Size:        {size_mb:.2f} MB")
    
    if info.get('models_profiled'):
        print(f"Models:      {', '.join(info['models_profiled'][:5])}...")
    
    print(f"\nInstalled:   {'Yes' if info['installed'] else 'No'}")
    if info['installed']:
        print(f"Local path:  {info['local_path']}")
else:
    print("Package not found")

In [ ]:
# Download weights from HuggingFace Hub
# This downloads pre-trained routing weights

try:
    # Download the MMLU weights
    weights_path = download("weights-mmlu-v1", quiet=False)
    print(f"\nWeights installed at: {weights_path}")
except Exception as e:
    print(f"Could not download weights: {e}")
    print("\nNote: Weights need to be uploaded to HuggingFace Hub first.")
    print("Repository: pureai-ecosystem/lunar-router-weights")
    print("\nTo upload weights:")
    print("  huggingface-cli login")
    print("  huggingface-cli upload pureai-ecosystem/lunar-router-weights ./dist/weights-mmlu-v1.zip")

In [ ]:
# Check weights installation path
weights_path = path("weights-mmlu-v1")
print(f"Weights path: {weights_path}")
print(f"Installed: {hub.is_installed('weights-mmlu-v1')}")

if weights_path.exists():
    # List contents
    print("\nContents:")
    for item in sorted(weights_path.iterdir()):
        if item.is_dir():
            files = list(item.iterdir())
            print(f"  {item.name}/ ({len(files)} files)")
            for f in files[:3]:
                print(f"    - {f.name}")
            if len(files) > 3:
                print(f"    ... and {len(files) - 3} more")
        else:
            print(f"  {item.name}")

## 3. Provider Overview

OpenTracy supports 7 LLM providers with 44+ pre-configured models.

In [ ]:
providers = {
    "OpenAI": OpenAIClient.COSTS,
    "Anthropic": AnthropicClient.COSTS,
    "Google Gemini": GoogleClient.COSTS,
    "Groq": GroqClient.COSTS,
    "Mistral": MistralClient.COSTS,
}

print("LUNAR ROUTER - AVAILABLE PROVIDERS")
print("=" * 60)

total_models = 0
for provider_name, models in providers.items():
    print(f"\n{provider_name} ({len(models)} models)")
    print("-" * 40)
    for model, cost in list(models.items())[:5]:  # Show first 5
        cost_str = f"${cost*1000:.4f}/1M tokens" if cost > 0 else "Free"
        print(f"  {model}: {cost_str}")
    if len(models) > 5:
        print(f"  ... and {len(models) - 5} more models")
    total_models += len(models)

print("\n" + "=" * 60)
print(f"Total: {len(providers)} providers, {total_models} models")
print("+ vLLM (local models) + Mock (testing)")

## 4. Testing LLM Clients

### 4.1 Mock Client (No API key needed)

In [ ]:
# Create a mock client for testing
mock = create_client("mock", "test-model")
print(f"Created: {mock}")

# Generate a response
response = mock.generate("What is 2+2?", max_tokens=50)
print(f"\nResponse: {response.text}")
print(f"Tokens: {response.tokens_used}")
print(f"Latency: {response.latency_ms:.2f}ms")

### 4.2 OpenAI Client

In [ ]:
# Test OpenAI (requires OPENAI_API_KEY)
if os.environ.get("OPENAI_API_KEY"):
    openai_client = create_client("openai", "gpt-4o-mini")
    print(f"Created: {openai_client}")
    
    response = openai_client.generate(
        "What is the capital of France? Answer in one word.",
        max_tokens=10,
        temperature=0.0
    )
    print(f"\nResponse: {response.text}")
    print(f"Tokens: {response.tokens_used} (in: {response.input_tokens}, out: {response.output_tokens})")
    print(f"Latency: {response.latency_ms:.2f}ms")
else:
    print("OPENAI_API_KEY not set. Skipping OpenAI test.")

### 4.3 Anthropic Client

In [ ]:
# Test Anthropic (requires ANTHROPIC_API_KEY)
if os.environ.get("ANTHROPIC_API_KEY"):
    anthropic_client = create_client("anthropic", "claude-3-5-haiku-20241022")
    print(f"Created: {anthropic_client}")
    
    response = anthropic_client.generate(
        "What is 15 * 23? Just give the number.",
        max_tokens=10,
        temperature=0.0
    )
    print(f"\nResponse: {response.text}")
    print(f"Tokens: {response.tokens_used}")
    print(f"Latency: {response.latency_ms:.2f}ms")
else:
    print("ANTHROPIC_API_KEY not set. Skipping Anthropic test.")

### 4.4 Google Gemini Client

In [ ]:
# Test Google Gemini (requires GOOGLE_API_KEY)
if os.environ.get("GOOGLE_API_KEY"):
    google_client = create_client("google", "gemini-1.5-flash")
    print(f"Created: {google_client}")
    
    response = google_client.generate(
        "Name three programming languages. Just list them.",
        max_tokens=30,
        temperature=0.0
    )
    print(f"\nResponse: {response.text}")
    print(f"Tokens: {response.tokens_used}")
    print(f"Latency: {response.latency_ms:.2f}ms")
else:
    print("GOOGLE_API_KEY not set. Skipping Google test.")

### 4.5 Groq Client (Ultra-fast inference)

In [ ]:
# Test Groq (requires GROQ_API_KEY)
if os.environ.get("GROQ_API_KEY"):
    groq_client = create_client("groq", "llama-3.1-8b-instant")
    print(f"Created: {groq_client}")
    
    response = groq_client.generate(
        "What is machine learning in one sentence?",
        max_tokens=50,
        temperature=0.0
    )
    print(f"\nResponse: {response.text}")
    print(f"Tokens: {response.tokens_used}")
    print(f"Latency: {response.latency_ms:.2f}ms (Groq is fast!)")
else:
    print("GROQ_API_KEY not set. Skipping Groq test.")

### 4.6 Mistral Client

In [ ]:
# Test Mistral (requires MISTRAL_API_KEY)
if os.environ.get("MISTRAL_API_KEY"):
    mistral_client = create_client("mistral", "mistral-small-latest")
    print(f"Created: {mistral_client}")
    
    response = mistral_client.generate(
        "Write a haiku about coding.",
        max_tokens=50,
        temperature=0.7
    )
    print(f"\nResponse:\n{response.text}")
    print(f"\nTokens: {response.tokens_used}")
    print(f"Latency: {response.latency_ms:.2f}ms")
else:
    print("MISTRAL_API_KEY not set. Skipping Mistral test.")

## 5. Semantic Router

The router uses pre-trained embeddings to select the best model for each prompt.

In [ ]:
# Try to load the router
try:
    router = load_router(verbose=True)
    print(f"\nRouter loaded!")
    print(f"   Clusters: {router.cluster_assigner.num_clusters}")
    print(f"   Models: {len(router.registry)}")
except FileNotFoundError as e:
    print(f"Could not load router: {e}")
    print("\nDownload weights first:")
    print("  CLI:    opentracy download weights-mmlu-v1")
    print("  Python: opentracy.download('weights-mmlu-v1')")
    router = None

In [ ]:
if router:                                                                                                                   
      test_prompts = [                                                                                                         
          "What is the capital of France?",                                                                                    
          "Write a Python function to sort a list.",                                                                           
          "Explain quantum entanglement in simple terms.",                                                                     
          "Translate 'hello' to Spanish.",                                                                                     
          "What are the symptoms of the flu?",                                                                                 
      ]                                                                                                                        
                                                                                                                               
      print("Routing test prompts:")                                                                                           
      print("=" * 60)                                                                                                          
                                                                                                                               
      for prompt in test_prompts:                                                                                              
          decision = router.route(prompt)                                                                                      
          print(f"\nPrompt: {prompt[:50]}")                                                                                    
          print(f"   -> Model: {decision.selected_model}")                                                                     
          print(f"   -> Expected Error: {decision.expected_error:.4f}, Cluster: {decision.cluster_id}")                        
else:                                                                                                                        
      print("Router not available. Download weights first.")          

In [ ]:
# Test routing with cost weight (if router is available)
if router:
    print("Routing with different cost weights:")
    print("=" * 60)
    print("(Higher cost_weight = prefer cheaper models)")
    
    prompt = "Explain the theory of relativity."
    print(f"\nPrompt: {prompt}\n")
    
    for cost_weight in [0.0, 0.3, 0.5, 0.7, 1.0]:
        router.cost_weight = cost_weight
        decision = router.route(prompt)
        print(f"cost_weight={cost_weight:.1f} -> {decision.selected_model}")

## 6. Training Pipeline

You can train your own router with custom data.

In [ ]:
# Import training components
try:
    from opentracy import (
        train_clusters,
        profile_models,
        full_training_pipeline,
        TrainingConfig,
        PromptDataset,
        KMeansTrainer,
        PromptEmbedder,
        SentenceTransformerProvider,
    )
    print("Training modules available!")
    TRAINING_AVAILABLE = True
except ImportError as e:
    print(f"Training modules not available: {e}")
    print("Install with: pip install opentracy[training,embeddings]")
    TRAINING_AVAILABLE = False

In [ ]:
# Create a small sample dataset for demonstration
if TRAINING_AVAILABLE:
    sample_data = [
        {"prompt": "What is 2+2?", "ground_truth": "4"},
        {"prompt": "What is the capital of Japan?", "ground_truth": "Tokyo"},
        {"prompt": "Who wrote Romeo and Juliet?", "ground_truth": "Shakespeare"},
        {"prompt": "What is H2O?", "ground_truth": "Water"},
        {"prompt": "What planet is known as the Red Planet?", "ground_truth": "Mars"},
        {"prompt": "What is the largest mammal?", "ground_truth": "Blue whale"},
        {"prompt": "How many continents are there?", "ground_truth": "7"},
        {"prompt": "What is the speed of light?", "ground_truth": "299,792,458 m/s"},
        {"prompt": "Who painted the Mona Lisa?", "ground_truth": "Leonardo da Vinci"},
        {"prompt": "What is Python?", "ground_truth": "A programming language"},
    ]
    
    dataset = PromptDataset(sample_data)
    print(f"Created sample dataset with {len(dataset)} prompts")

In [ ]:
# Test embedder
if TRAINING_AVAILABLE:
    print("Loading embedder (this may take a moment on first run)...")
    provider = SentenceTransformerProvider(model_name="all-MiniLM-L6-v2")
    embedder = PromptEmbedder(provider, cache_enabled=True)
    
    # Test embedding
    test_prompt = "What is machine learning?"
    embedding = embedder.embed(test_prompt)
    
    print(f"Embedder loaded!")
    print(f"   Model: all-MiniLM-L6-v2")
    print(f"   Embedding dimension: {embedding.shape[0]}")
    print(f"   Test prompt: '{test_prompt}'")
    print(f"   Embedding (first 5 values): {embedding[:5]}")

In [ ]:
# Train clusters (demonstration with small K)
if TRAINING_AVAILABLE:
    print("Training clusters...")
    
    trainer = KMeansTrainer(embedder, num_clusters=3)  # Small K for demo
    cluster_assigner = trainer.train(dataset, verbose=True)
    
    print(f"\nClusters trained!")
    print(f"   Number of clusters: {cluster_assigner.num_clusters}")
    print(f"   Centroid shape: {cluster_assigner.centroids.shape}")

In [ ]:
# Test cluster assignment
if TRAINING_AVAILABLE:
    print("Testing cluster assignments:")
    print("=" * 50)
    
    for prompt, _ in dataset:
        embedding = embedder.embed(prompt)
        result = cluster_assigner.assign(embedding)
        print(f"Cluster {result.cluster_id}: {prompt[:40]}")

## 7. Compare Providers

Let's compare responses from different providers on the same prompt.

In [ ]:
# Collect available clients
available_clients = []

# Always add mock
available_clients.append(("Mock", create_client("mock", "mock-model")))

# Add real providers if API keys are available
if os.environ.get("OPENAI_API_KEY"):
    available_clients.append(("OpenAI", create_client("openai", "gpt-4o-mini")))

if os.environ.get("ANTHROPIC_API_KEY"):
    available_clients.append(("Anthropic", create_client("anthropic", "claude-3-5-haiku-20241022")))

if os.environ.get("GOOGLE_API_KEY"):
    available_clients.append(("Google", create_client("google", "gemini-1.5-flash")))

if os.environ.get("GROQ_API_KEY"):
    available_clients.append(("Groq", create_client("groq", "llama-3.1-8b-instant")))

if os.environ.get("MISTRAL_API_KEY"):
    available_clients.append(("Mistral", create_client("mistral", "mistral-small-latest")))

print(f"Available clients: {len(available_clients)}")
for name, _ in available_clients:
    print(f"  - {name}")

In [ ]:
# Compare providers
if len(available_clients) > 1:
    test_prompt = "What is the meaning of life? Answer in one sentence."
    
    print(f"Prompt: {test_prompt}")
    print("=" * 70)
    
    for name, client in available_clients:
        try:
            response = client.generate(test_prompt, max_tokens=100, temperature=0.0)
            resp_text = response.text[:150] + "..." if len(response.text) > 150 else response.text
            print(f"\n{name}:")
            print(f"  {resp_text}")
            print(f"  [{response.latency_ms:.0f}ms, {response.tokens_used} tokens]")
        except Exception as e:
            print(f"\n{name}: Error - {e}")
else:
    print("Set API keys to compare multiple providers.")

## 8. Summary

This notebook demonstrated:

1. **Hub System**: Download manager like NLTK/spaCy/HuggingFace
2. **7 LLM Providers**: OpenAI, Anthropic, Google, Groq, Mistral, vLLM, Mock
3. **44+ Models**: Pre-configured with pricing
4. **Semantic Routing**: Route prompts to the best model
5. **Training Pipeline**: Train custom routers

### CLI Quick Reference

```bash
# Package management
opentracy list                      # List packages
opentracy download weights-mmlu-v1  # Download
opentracy info weights-mmlu-v1      # Package info
opentracy path weights-mmlu-v1      # Show path
opentracy remove weights-mmlu-v1    # Remove
opentracy verify weights-mmlu-v1    # Verify
```

### Python Quick Reference

```python
import opentracy

# Download weights from HuggingFace
opentracy.download("weights-mmlu-v1")

# Load router
router = opentracy.load_router()
decision = router.route("Your prompt here")
print(f"Use model: {decision.selected_model}")

# Create LLM client
client = opentracy.create_client("openai", "gpt-4o-mini")
response = client.generate("Hello!")
```

In [ ]:
print("OpenTracy Test Complete!")
print(f"Version: {__version__}")
print(f"Data directory: {OPENTRACY_DATA_HOME}")
print("\nLinks:")
print("  GitHub: https://github.com/pureai-ecosystem/opentracy")
print("  Weights: https://huggingface.co/pureai-ecosystem/lunar-router-weights")